# Подготовка

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

Mounted at /content/drive


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path
import numpy as np
import torch.nn.functional as F
import shutil
import librosa
import soundfile as sf
import tempfile
import random

# RuBert

In [3]:
!pip install transformers==4.44.0 sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
model_path = "/content/drive/MyDrive/rubert_tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_path)

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
class RuBERTClassifier(nn.Module):
    def __init__(self, encoder, num_classes=7, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(312, num_classes)  # 312 — размер эмбеддингов rubert-tiny2

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Берём CLS-токен (как пулинг в GigaAM-Emo)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_embedding)
        logits = self.classifier(x)
        return logits

In [4]:
encoder = AutoModel.from_pretrained(model_path)
full_model = RuBERTClassifier(encoder, num_classes=7)
full_model.load_state_dict(torch.load("/content/drive/MyDrive/rubert_semantic_best.pth"))
full_model.eval()
encoder = full_model.encoder

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In [5]:
data_dir = Path("/content/drive/MyDrive/876_augmented/")
txt_path = data_dir / "ASR.txt"

texts = []
names = []
with open(txt_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            name, text = parts
            texts.append(text)
            names.append(name)

In [6]:
class EmbeddingDataset(Dataset):
    def __init__(self, texts, names, tokenizer, max_len=256):
        self.texts = texts
        self.names = names
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        name = self.names[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'name': name
        }

embedding_dataset = EmbeddingDataset(texts, names, tokenizer)
embedding_loader = DataLoader(embedding_dataset, batch_size=32, shuffle=False)

In [7]:
OUTPUT = "/content/drive/MyDrive/876_augmented/RuBert_Emb.txt"

with open(OUTPUT, 'w', encoding='utf-8') as f:
    for batch in embedding_loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        names = batch['name']

        outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]

        for i, name in enumerate(names):
            emb_str = ','.join([str(x.item()) for x in cls_embedding[i]])
            f.write(f"{name} {emb_str}\n")

# Emo

In [8]:
drive_repo_path = "/content/drive/MyDrive/GigaAM_repo"
repo_path = "/content/GigaAM"
shutil.copytree(drive_repo_path, repo_path, dirs_exist_ok=True)
%cd {repo_path}
!pip install -e .

print("Библиотека восстановлена")

/content/GigaAM
Obtaining file:///content/GigaAM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.9 MB/s eta 0:00:00
  Building editable for gigaa

Библиотека восстановлена


In [2]:
import gigaam
import time

print("Загружаем GigaAM-Emo...")
start_load = time.time()
model_emo = gigaam.load_model("emo").float()
print(f"Загружена за {time.time() - start_load:.2f} сек")
new_head = nn.Linear(768, 6)
model_emo.head = new_head
model_emo.load_state_dict(torch.load("/content/drive/MyDrive/emo_model_6classes_weights.pth"))
model_emo.eval()

Загружаем GigaAM-Emo...


100%|███████████████████████████████████████| 462M/462M [00:12<00:00, 38.7MiB/s]


Загружена за 17.27 сек


GigaAMEmo(
  (preprocessor): FeatureExtractor(
    (featurizer): Sequential(
      (0): MelSpectrogram(
        (spectrogram): Spectrogram()
        (mel_scale): MelScale()
      )
      (1): SpecScaler()
    )
  )
  (encoder): ConformerEncoder(
    (pre_encode): StridingSubsampling(
      (out): Linear(in_features=12288, out_features=768, bias=True)
      (conv): Sequential(
        (0): Conv2d(1, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): ReLU()
        (2): Conv2d(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (3): ReLU()
      )
    )
    (pos_enc): RelPositionalEmbedding()
    (layers): ModuleList(
      (0-15): 16 x ConformerLayer(
        (norm_feed_forward1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (feed_forward1): ConformerFeedForward(
          (linear1): Linear(in_features=768, out_features=3072, bias=True)
          (activation): SiLU()
          (linear2): Linear(in_features=3072, out_features=768, bias=

In [3]:
OUTPUT = "/content/drive/MyDrive/876_augmented/Emo_Emb.txt"
folder_path = "/content/drive/MyDrive/876_augmented/"

with open(OUTPUT, 'w', encoding='utf-8') as fl:
  with torch.no_grad():
    for f in Path(folder_path).iterdir():
      if f.suffix in ['.wav', '.mp3']:
        embedding = model_emo.get_embedding(str(f))
        name = f.stem
        emb_str = ','.join([str(x.item()) for x in embedding])
        fl.write(f"{name} {emb_str}\n")